In [1]:
# Hotel Pricing Lakehouse Project - Silver Cleaning

import pandas as pd
from pathlib import Path
from datetime import datetime

# Define paths
BRONZE_PATH = Path("C:\\Users\\cxpen\\Documents\\JOB\\Portfolios\\hotel-pricing-lakehouse\\data\\bronze\\hotel_bookings_bronze.csv")
SILVER_DIR = Path("C:\\Users\\cxpen\\Documents\\JOB\\Portfolios\\hotel-pricing-lakehouse\\data\\silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# Read Bronze data
df_bronze = pd.read_csv(BRONZE_PATH)

print("Bronze shape:", df_bronze.shape)

Bronze shape: (119390, 35)


In [2]:
df_silver = df_bronze.copy()

In [3]:
before_rows = len(df_silver)

df_silver = df_silver.drop_duplicates()

after_rows = len(df_silver)

print("Duplicate rows removed:", before_rows - after_rows)

Duplicate rows removed: 31994


In [6]:
df_silver.head()
df_silver.duplicated().sum()

0

In [ ]:
# full info
df_silver.info(
    verbose=True,
    show_counts=True,
    max_cols=100
)



<class 'pandas.core.frame.DataFrame'>
Int64Index: 87396 entries, 0 to 119389
Data columns (total 35 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87396 non-null  object 
 1   is_canceled                     87396 non-null  int64  
 2   lead_time                       87396 non-null  int64  
 3   arrival_date_year               87396 non-null  int64  
 4   arrival_date_month              87396 non-null  object 
 5   arrival_date_week_number        87396 non-null  int64  
 6   arrival_date_day_of_month       87396 non-null  int64  
 7   stays_in_weekend_nights         87396 non-null  int64  
 8   stays_in_week_nights            87396 non-null  int64  
 9   adults                          87396 non-null  int64  
 10  children                        87392 non-null  float64
 11  babies                          87396 non-null  int64  
 12  meal                           

In [38]:
df_silver = df_silver[df_silver["adr"] >= 0].copy()
df_silver.shape


(87229, 38)

In [39]:
df_silver["total_guests"] = (
    df_silver["adults"] + 
    df_silver["children"].fillna(0) + 
    df_silver["babies"]
)
# Filter rows where total_guests is 0
zero_guest_rows = df_silver[df_silver["total_guests"] == 0]

# Print number of rows
print("Number of zero-guest rows:", zero_guest_rows.shape[0])

# Display full rows
display(zero_guest_rows)


Number of zero-guest rows: 0


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,ingestion_timestamp,source_file_name,batch_id,total_guests,arrival_date,total_nights


In [40]:
df_silver = df_silver[df_silver["total_guests"] > 0]    

In [41]:
df_silver["country"] = df_silver["country"].fillna("Unknown")

In [42]:
# children should be 0 if missing
df_silver["children"] = df_silver["children"].fillna(0)

# country can be Unknown if missing
df_silver["country"] = df_silver["country"].fillna("Unknown")

# agent and company have many missing values, keep them for now but replace missing with 0
df_silver["agent"] = df_silver["agent"].fillna(0)
df_silver["company"] = df_silver["company"].fillna(0)

In [43]:
df_silver["arrival_date"] = pd.to_datetime(
    df_silver["arrival_date_year"].astype(str) + "-" +
    df_silver["arrival_date_month"] + "-" +
    df_silver["arrival_date_day_of_month"].astype(str),
    errors="coerce"
)

In [44]:
df_silver["total_nights"] = (
    df_silver["stays_in_weekend_nights"] +
    df_silver["stays_in_week_nights"]
)

In [45]:
df_silver["total_guests"] = (
    df_silver["adults"] +
    df_silver["children"] +
    df_silver["babies"]
)

In [46]:
df_silver["booking_status"] = df_silver["is_canceled"].map({
    0: "Not Canceled",
    1: "Canceled"
})

In [47]:
df_silver["estimated_revenue"] = df_silver["adr"] * df_silver["total_nights"]

In [48]:
before_cleaning = len(df_silver)

df_silver = df_silver[
    (df_silver["total_guests"] > 0) &
    (df_silver["total_nights"] > 0) &
    (df_silver["adr"] >= 0) &
    (df_silver["arrival_date"].notna())
]

after_cleaning = len(df_silver)

print("Invalid rows removed:", before_cleaning - after_cleaning)
print("Silver shape:", df_silver.shape)

Invalid rows removed: 591
Silver shape: (86638, 40)


In [49]:
df_silver["silver_processed_timestamp"] = datetime.now()

In [53]:
# Convert meal code into readable meal plan name

meal_mapping = {
    "BB": "Bed and Breakfast",
    "HB": "Half Board",
    "FB": "Full Board",
    "SC": "Self Catering",
    "Undefined": "Undefined / No Meal Package"
}

df_silver["meal_plan"] = df_silver["meal"].map(meal_mapping)

In [57]:
SILVER_PATH = SILVER_DIR / "hotel_bookings_silver.csv"

df_silver.to_csv(SILVER_PATH, index=False)

print("Silver cleaning completed.")
print(f"Silver file saved to: {SILVER_PATH}")
print("Final Silver shape:", df_silver.shape)

Silver cleaning completed.
Silver file saved to: C:\Users\cxpen\Documents\JOB\Portfolios\hotel-pricing-lakehouse\data\silver\hotel_bookings_silver.csv
Final Silver shape: (86638, 42)


In [51]:
df_silver.isnull().sum().sort_values(ascending=False).head(10)

hotel                          0
booking_changes                0
agent                          0
company                        0
days_in_waiting_list           0
customer_type                  0
adr                            0
required_car_parking_spaces    0
total_of_special_requests      0
reservation_status             0
dtype: int64

In [52]:
df_silver[
    [
        "arrival_date",
        "total_nights",
        "total_guests",
        "booking_status",
        "estimated_revenue"
    ]
].head()

,arrival_date,total_nights,total_guests,booking_status,estimated_revenue
2,2015-07-01,1,1.0,Not Canceled,75.0
3,2015-07-01,1,1.0,Not Canceled,75.0
4,2015-07-01,2,2.0,Not Canceled,196.0
6,2015-07-01,2,2.0,Not Canceled,214.0
7,2015-07-01,2,2.0,Not Canceled,206.0


,meal,meal_plan
2,BB,Bed and Breakfast
3,BB,Bed and Breakfast
4,BB,Bed and Breakfast
6,BB,Bed and Breakfast
7,FB,Full Board
